In [ ]:
# 1. df_nav

import pandas as pd

df_nav = pd.read_csv("../data/df_nav.csv", parse_dates=["datadate", "announcement_date"]).copy()
df_nav.loc[:, "quarter"] = pd.PeriodIndex(df_nav["datadate"], freq="Q")

print("Date range:")
print(df_nav["datadate"].agg(["min", "max"]))

print("\nTickers:", sorted(df_nav["ticker"].dropna().unique()))
print("\nQuarter counts by ticker:")
print(df_nav.groupby("ticker")["quarter"].nunique())

print("\nDuplicate ticker-quarter rows:")
print(df_nav.duplicated(["ticker", "quarter"]).sum())

print("\nMissing announcement dates by ticker:")
print(df_nav.groupby("ticker")["announcement_date"].apply(lambda s: s.isna().sum()))

print("\nAnnouncement date before datadate:")
print((df_nav["announcement_date"] < df_nav["datadate"]).sum())

print("\nAnnouncement lag (days) by ticker:")
lag_days = (df_nav["announcement_date"] - df_nav["datadate"]).dt.days
print(lag_days.groupby(df_nav["ticker"]).describe()[["count", "min", "50%", "max"]])

In [ ]:
# 2. df_trade

import pandas as pd

df_trade = pd.read_csv("../data/df_trade_daily.csv", parse_dates=["trade_date"]).copy()

print("Date range:")
print(df_trade["trade_date"].agg(["min", "max"]))

print("\nTickers:", sorted(df_trade["ticker"].dropna().unique()))
print("\nObservations by ticker:")
print(df_trade.groupby("ticker")["trade_date"].count())

print("\nDuplicate ticker-trade_date rows:")
print(df_trade.duplicated(["ticker", "trade_date"]).sum())

print("\nMissing values:")
print(df_trade[["ticker", "trade_date", "price", "ret"]].isna().sum())

print("\nNonpositive prices:")
print((df_trade["price"] <= 0).sum())

print("\nMissing returns:")
print(df_trade["ret"].isna().sum())


In [ ]:
# 3. df_rate

import pandas as pd

df_rate = pd.read_csv("../data/df_rate_daily.csv", parse_dates=["rate_date"])

print("Date range:")
print(df_rate["rate_date"].agg(["min", "max"]))

print("\nDuplicate rate_date rows:")
print(df_rate.duplicated(["rate_date"]).sum())

print("\nMissing values:")
print(df_rate[["rate_date", "dgs10", "dbaa", "spread"]].isna().sum())

print("\nSpread consistency check:")
print((df_rate["spread"] - (df_rate["dbaa"] - df_rate["dgs10"])).abs().max())


In [ ]:
df_nav.columns.to_list()
df_trade.columns.to_list()
df_rate.columns.to_list()

In [ ]:
# study 2016-01-01 to 2024-12-31

import pandas as pd

trade = df_trade.copy(deep=True)
nav = df_nav.copy(deep=True)
rate = df_rate.copy(deep=True)

trade.loc[:, "trade_date"] = pd.to_datetime(trade["trade_date"], errors="coerce")
nav.loc[:, "announcement_date"] = pd.to_datetime(nav["announcement_date"], errors="coerce")
rate.loc[:, "rate_date"] = pd.to_datetime(rate["rate_date"], errors="coerce")

trade = trade[
    (trade["trade_date"] >= "2016-01-01") &
    (trade["trade_date"] <= "2024-12-31")
].copy()




In [52]:
# rate forward-fill -> df_trade_rate

rate_ffill = (
    df_rate.copy()
    .sort_values("rate_date")
    .drop_duplicates("rate_date")
    .set_index("rate_date")
    .asfreq("D")
)

rate_ffill.loc[:, ["dgs10", "dbaa"]] = rate_ffill[["dgs10", "dbaa"]].ffill()
rate_ffill.loc[:, "spread"] = rate_ffill["dbaa"] - rate_ffill["dgs10"]
rate_ffill = rate_ffill.reset_index()

# merge with forward-filled rates

df_trade_rate = trade.merge(
    rate_ffill[["rate_date", "dgs10", "dbaa", "spread"]],
    left_on="trade_date",
    right_on="rate_date",
    how="left"
)
df_trade_rate.head()


,trade_date,permno,ticker,price,ret,vol,shrout,rate_date,dgs10,dbaa,spread
0,2016-01-04,90401,ARCC,14.46,0.014737,2784659.0,314347.0,2016-01-04,2.24,5.48,3.24
1,2016-01-05,90401,ARCC,14.59,0.008990,1483387.0,314347.0,2016-01-05,2.25,5.50,3.25
2,2016-01-06,90401,ARCC,14.54,-0.003427,2168723.0,314347.0,2016-01-06,2.18,5.44,3.26
3,2016-01-07,90401,ARCC,14.08,-0.031637,2210695.0,314347.0,2016-01-07,2.16,5.44,3.28
4,2016-01-08,90401,ARCC,13.91,-0.012074,1902011.0,314347.0,2016-01-08,2.13,5.44,3.31


In [53]:
# trade_rate + nav

df = pd.merge_asof(
    df_trade_rate.sort_values(["trade_date", "ticker"]).reset_index(drop=True),
    df_nav[["ticker", "announcement_date", "nav", "nav_ret"]]
        .sort_values(["announcement_date", "ticker"])
        .reset_index(drop=True),
    left_on="trade_date",
    right_on="announcement_date",
    by="ticker",
    direction="backward"
)

In [66]:
# event study: -1 to +3 days around announcement

import duckdb as dd

df_event_study = dd.query("""
    with events as (
        select distinct ticker, announcement_date
        from df
    ),
    pre as (
        select
            d.*,
            e.announcement_date as event_date,
            - row_number() over (
                partition by d.ticker, e.announcement_date
                order by d.trade_date desc
            ) as event_day
        from events e
        join df d
          on d.ticker = e.ticker
         and d.trade_date < e.announcement_date
         and d.trade_date >= e.announcement_date - interval 7 day
    ),
    on_day as (
        select
            d.*,
            e.announcement_date as event_date,
            0 as event_day
        from events e
        join df d
          on d.ticker = e.ticker
         and d.trade_date = e.announcement_date
    ),
    post as (
        select
            d.*,
            e.announcement_date as event_date,
            row_number() over (
                partition by d.ticker, e.announcement_date
                order by d.trade_date
            ) as event_day
        from events e
        join df d
          on d.ticker = e.ticker
         and d.trade_date > e.announcement_date
         and d.trade_date <= e.announcement_date + interval 7 day
    )
    select *
    from (
        select * from pre
        union all
        select * from on_day
        union all
        select * from post
    )
    where event_day between -1 and 3
    order by ticker, event_date, trade_date
""").to_df()

df_event_study.to_csv("../data/df_event_study.csv", index=False)
df_event_study.head()


,trade_date,permno,ticker,price,ret,vol,shrout,rate_date,dgs10,dbaa,spread,announcement_date,nav,nav_ret,event_date,event_day
0,2016-02-23,90401,ARCC,12.92,0.007015,1505792.0,314347.0,2016-02-23,1.74,5.32,3.58,2015-11-04,16.789579,-0.000500,2016-02-24,-1
1,2016-02-24,90401,ARCC,12.92,0.000000,2100705.0,314347.0,2016-02-24,1.75,5.31,3.56,2016-02-24,16.457393,-0.019785,2016-02-24,0
2,2016-02-25,90401,ARCC,13.39,0.036378,2749160.0,314347.0,2016-02-25,1.71,5.27,3.56,2016-02-24,16.457393,-0.019785,2016-02-24,1
3,2016-02-26,90401,ARCC,13.51,0.008962,2112166.0,314347.0,2016-02-26,1.76,5.32,3.56,2016-02-24,16.457393,-0.019785,2016-02-24,2
4,2016-02-29,90401,ARCC,13.66,0.011103,2084226.0,314347.0,2016-02-29,1.74,5.29,3.55,2016-02-24,16.457393,-0.019785,2016-02-24,3


In [68]:
# validate df_event_study

import pandas as pd

df_event_study = pd.read_csv(
    "../data/df_event_study.csv",
    parse_dates=["trade_date", "announcement_date", "event_date", "rate_date"]
).copy()

event_days = df_event_study.groupby(["ticker", "event_date"])["event_day"].agg(list)

print("Null/NaN dates:")
print(df_event_study[["trade_date", "announcement_date", "event_date", "rate_date"]].isna().sum())

print("\nEvent-day values present:")
print(sorted(df_event_study["event_day"].dropna().unique()))

print("\nEvents by ticker:")
print(df_event_study[["ticker", "event_date"]].drop_duplicates().groupby("ticker").size())

print("\nEvents missing full [-1,0,1,2,3]:")
bad_events = event_days[event_days.apply(lambda x: sorted(x) != [-1, 0, 1, 2, 3])]
print(len(bad_events))
print(bad_events.head(20))

print("\nEvents missing -1 baseline:")
missing_baseline = event_days[event_days.apply(lambda x: -1 not in x)]
print(len(missing_baseline))
print(missing_baseline.head(20))

print("\nDuplicate ticker-event_date-event_day rows:")
print(df_event_study.duplicated(["ticker", "event_date", "event_day"]).sum())


Null/NaN dates:
trade_date           0
announcement_date    0
event_date           0
rate_date            0
dtype: int64

Event-day values present:
[np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Events by ticker:
ticker
ARCC    36
MAIN    36
dtype: int64

Events missing full [-1,0,1,2,3]:
0
Series([], Name: event_day, dtype: object)

Events missing -1 baseline:
0
Series([], Name: event_day, dtype: object)

Duplicate ticker-event_date-event_day rows:
0
